In [25]:
import math

mod1 = {
    'Potência': 725,
    'Qtd.': 140832,
    'Strings': 4401,
    'Vmp': 41,
    'Voc': 49.2,
    'Imp': 17.69,
    'Isc': 18.74
}

mod2 = {
    'Potência': 730,
    'Qtd.': 254912,
    'Strings': 7966,
    'Vmp': 41.11,
    'Voc': 49.36,
    'Imp': 17.76,
    'Isc': 18.81
}

inv = {
    'Max Input Voltage': 1500,
    'No. of MPPTs': 6,
    'Max Current per MPPT': 106,
    'Max Short Circuit per MPPT': 180,
    'Max PV Inputs per MPPT': 6
}

inverters = 464
PSTs = 29
inverters_per_PST = inverters / PSTs
if inverters_per_PST % 1 != 0:
    inverter_per_PST_up = math.ceil(inverters_per_PST)
    inverter_per_PST_down = math.floor(inverters_per_PST)
    print('Inverters per PST is not a multiple of 1.')
    print(f'The number of inverters per PSTs might be: {inverter_per_PST_up} and {inverter_per_PST_down}')
else:
    print(f'The number of inverters per PST might be: {inverters_per_PST}')

The number of inverters per PST might be: 16.0


In [26]:
for i in range(inv['Max PV Inputs per MPPT']):
    sc_current = i*mod2['Isc']
    mp_current = i*mod2['Imp']
    max_number_of_PV_ISC = i
    max_number_of_PV_MP = i

    if sc_current > inv['Max Short Circuit per MPPT']:
        sc_current = (i-1)*mod2['Isc']
        max_number_of_PV_ISC = (i-1)
    if mp_current > inv['Max Current per MPPT']:
        mp_current = (i-1)*mod2['Imp']
        max_number_of_PV_MP = (i-1)

max_number_of_PV = min(max_number_of_PV_MP, max_number_of_PV_ISC, inv['Max PV Inputs per MPPT'])

print(f'The max number of strings per MPPT is: {max_number_of_PV}')
print(f'The max number of strings per inverter is: {max_number_of_PV*inv['No. of MPPTs']}')

strings_per_inverter = (mod1['Strings']+mod2['Strings'])/inverters
strings_per_inverter_up = math.ceil(strings_per_inverter)
if strings_per_inverter_up != strings_per_inverter:
    print('\nStrings per inverter is not a multiple of 1.')
    print(f'The number of strings per inverter might be: {strings_per_inverter_up} and {strings_per_inverter_up-1}')
else:
    print(f'The number of strings per inverter might be: {strings_per_inverter}')

The max number of strings per MPPT is: 5
The max number of strings per inverter is: 30

Strings per inverter is not a multiple of 1.
The number of strings per inverter might be: 27 and 26


In [27]:
i = 1
while True:
    string_voltage = i * mod2['Voc']
    if string_voltage > inv['Max Input Voltage']:
        string_voltage = (i-1)*mod2['Voc']
        modules_per_inverter = i-1
        break
    i += 1

modules_per_inverter = 32
print(f'The max number of modules per string is: {modules_per_inverter}')

The max number of modules per string is: 32


In [30]:
max_number_of_725_inverters = math.ceil(mod1['Strings']/strings_per_inverter)
for i in range(max_number_of_725_inverters):
    inv_26_725 = i
    inv_27_725 = max_number_of_725_inverters - i-1
    condition1 = inv_26_725*26 + inv_27_725*27 == mod1['Strings']
    condition2 = inv_26_725*26*modules_per_inverter + inv_27_725*27*modules_per_inverter == mod1['Qtd.']
    if condition1 and condition2:
        break

max_number_of_730_inverters = math.ceil(mod2['Strings']/strings_per_inverter)
for i in range(max_number_of_730_inverters):
    inv_26_730 = i
    inv_27_730 = max_number_of_730_inverters - i
    condition3 = inv_26_730*26 + inv_27_730*27 == mod2['Strings']
    condition4 = inv_26_730*26*modules_per_inverter + inv_27_730*27*modules_per_inverter == mod2['Qtd.']
    if condition3 and condition4:
        break

if condition1 and condition2 and condition3 and condition4:
    print(f'The number of inverters with module of 725Wp with 26 strings per inverter is: {inv_26_725}')
    print(f'The number of inverters with module of 725Wp with 27 strings per inverter is: {inv_27_725}')
    print(f'The number of inverters with module of 730Wp with 26 strings per inverter is: {inv_26_730}')
    print(f'The number of inverters with module of 730Wp with 27 strings per inverter is: {inv_27_730}')

The number of inverters with module of 725Wp with 26 strings per inverter is: 54
The number of inverters with module of 725Wp with 27 strings per inverter is: 111
The number of inverters with module of 730Wp with 26 strings per inverter is: 107
The number of inverters with module of 730Wp with 27 strings per inverter is: 192


In [67]:
import math

target = 371

def find_complete_strings(target):
    result = []
    # Check exact match first
    for i in range(math.ceil(target/26) + 1):
        for j in range(math.ceil(target/27) + 1):
            if i*26 + j*27 == target:
                result.append((i,j,target))
    if result:
        return result

    # If no exact match, search outward
    offset = 1
    while True:
        # Check higher
        for i in range(math.ceil((target+offset)/26) + 1):
            for j in range(math.ceil((target+offset)/27) + 1):
                if i*26 + j*27 == target + offset:
                    return i, j, target + offset

        # Check lower (if positive)
        if target - offset >= 0:
            for i in range(math.ceil((target-offset)/26) + 1):
                for j in range(math.ceil((target-offset)/27) + 1):
                    if i*26 + j*27 == target - offset:
                        return i, j, target - offset

        offset += 1

result = find_complete_strings(target)
# print(f"The closest number to {target} that works is: {result[2]}")
# print(f"Using {result[0]}×28 + {result[1]}×29 = {result[2]}")
# print(f'The difference between {target} and {result[2]} is {abs(target - result[2])}')
print(result)

[(7, 7, 371)]
